# 도시별 기온 예측 — 시계열 EDA와 모델 (Berkeley Earth)

- 데이터: 세계 100개 도시 월평균 기온 (239,177행)
- 목표: 도시별 월평균 기온 예측 — 다중 시계열
- 흐름: 불러오기 → 학습 전 확인 → 시계열 EDA → 형식 변환 → 학습 → 해석
- 참고: 데이터 소개 data_BerkeleyEarth_temperature.txt

- 이 데이터의 특징: **장기 추세 + 뚜렷한 계절성**
  · 서울 등 도시가 item_id → 다중 시계열
  · 남북반구 계절 반대 (전역 모델 논의)

## 1. 불러오기 · 도시 선택

- 원본 22만 행 · 100개 도시 → 실습용으로 일부 도시·기간 선택
- 계절성 대비가 큰 8개 도시 사용

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("GlobalLandTemperaturesByMajorCity.csv",
                 parse_dates=["dt"])
print(df.shape)          # (239177, 7)

cities = ["Seoul","Tokyo","London","New York",
          "Cairo","Singapore","Sydney","Rio De Janeiro"]
df = df[(df["City"].isin(cities)) & (df["dt"] >= "1900-01-01")]
df = df.dropna(subset=["AverageTemperature"])
print("선택 후:", df.shape)

## 2. 학습 전 확인 — 시계열 3요소로 정리

- 시계열은 item_id · timestamp · target 3열 형식이 기본
- 여기서 City=item_id, dt=timestamp, AverageTemperature=target

In [ ]:
sub = df.rename(columns={
    "City": "item_id",
    "dt": "timestamp",
    "AverageTemperature": "target",
})[["item_id", "timestamp", "target"]]
print(sub.head())
print("\n시계열 개수:", sub["item_id"].nunique())

### 2-1. 남북반구 계절 반대 (중요)

- 서울은 1월 최저·7월 최고, 시드니는 반대
- 전역 모델에 섞으면 계절 신호가 상충할 수 있음
- 먼저 계절 패턴을 눈으로 확인

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,3))
for city in ["Seoul", "Sydney", "Singapore"]:
    s = sub[sub["item_id"]==city]
    monthly = s.assign(m=s["timestamp"].dt.month).groupby("m")["target"].mean()
    ax.plot(monthly.index, monthly.values, marker="o", label=city)
ax.set_xlabel("month"); ax.set_ylabel("temp"); ax.legend()
ax.set_title("seasonal pattern by city")
plt.show()
# 서울↔시드니 반대, 싱가포르는 거의 평평(적도)

## 3. 시계열 EDA — 추이·계절성

- 시계열은 순서가 핵심 → 시간축 그래프로 본다

### 3-1. 특정 도시의 전체 추이

In [ ]:
seoul = sub[sub["item_id"]=="Seoul"].set_index("timestamp")["target"]

seoul.plot(figsize=(12,3), title="Seoul monthly temp (full)")
plt.show()
# 계절 진동 + 장기 추세 확인

### 3-2. 계절 분해 (추세·계절성·잔차)

- 강의자료 49p: 원본을 추세+계절성+잔차로 분리
- period=12 → 연 주기(12개월)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# 최근 20년만 (전체는 그래프가 빽빽)
recent = seoul.dropna().iloc[-240:]
result = seasonal_decompose(recent, model="additive", period=12)
result.plot()
plt.gcf().set_size_inches(11, 7)
plt.show()
# Seasonal 줄에서 연 주기가 뚜렷이 반복

## 4. 시계열 형식 변환

- TimeSeriesDataFrame으로 변환
- convert_frequency로 규칙적 간격(월초) 보장

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame

ts = TimeSeriesDataFrame.from_data_frame(
    sub, id_column="item_id", timestamp_column="timestamp")
ts = ts.convert_frequency(freq="MS")   # 월초 기준
print("변환 완료:", ts.shape)

## 5. 학습

- 시간 순 분할 (미래로 예측하는 구조)
- prediction_length = 12 (1년 앞)

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

prediction_length = 12
train_data, test_data = ts.train_test_split(prediction_length)

predictor = TimeSeriesPredictor(
    prediction_length=prediction_length,
    target="target",
    freq="MS",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)
# SeasonalNaive 순위 확인 — 계절성 뚜렷하면 상위에 옴

## 6. 해석

In [ ]:
predictions = predictor.predict(train_data)

# 서울 예측 시각화
predictor.plot(
    data=test_data,
    predictions=predictions,
    item_ids=["Seoul"],
    max_history_length=60,
)
plt.show()

### 6-1. 확인할 점 (강의자료 71p)

- 예측선이 계절 패턴을 따라가는가
- 실제값이 예측 구간(음영) 안에 들어오는가
- 도시별로 예측 난도가 다른가
  (싱가포르=쉬움, 계절 큰 도시=어려움)

## 정리

- 도시 = item_id → 다중 시계열
- 3열 형식 변환 · convert_frequency(월초)
- 계절 분해로 추세·계절성 확인 (49p)
- 남북반구 계절 반대 → 전역 모델 논의
- 시간 순 분할 · SeasonalNaive 순위 확인
- 학습은 TimeSeriesPredictor가 자동

- 정형과 다른 점: 순서가 핵심 · 시간 순 분할 · 예측 구간
  → 시계열은 "언제"가 중요하다